# **이진분류 실습: 이직 여부**

<center><img src = "https://github.com/Jangrae/img/blob/master/attrition.png?raw=true" width=800/></center>

## **1. 환경준비**

### (1) 라이브러리 불러오기

In [ ]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.backend import clear_session
from tensorflow.keras.optimizers import Adam
import warnings

warnings.filterwarnings(action='ignore')
%config InlineBackend.figure_format='retina'

### (2) 함수 만들기

In [ ]:
# 함수 만들기
def dl_history_plot(history):
    plt.figure(figsize=(10, 6))
    plt.plot(history['loss'], label='Train Loss', marker='.')
    plt.plot(history['val_loss'], label='Validation Loss', marker='.')

    plt.title('Learning Curve', size=15, pad=20)
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

### (3) 데이터 준비

In [ ]:
path = 'https://raw.githubusercontent.com/Jangrae/csv/master/Attrition_simple2.csv'
data = pd.read_csv(path)
data.head()

**데이터 설명**

- Attrition: 이직 여부 (1: 이직, 0: 잔류)
- Age: 나이
- DistanceFromHome: 집-직장 거리 (단위: 마일)
- EmployeeNumber: 사번
- Gender: 성별 (Male, Female)
- JobSatisfaction: 직무만족도(1: Low, 2: Medium, 3: High, 4: Very High)
- MaritalStatus: 결혼 상태 (Single, Married, Divorced)
- MonthlyIncome: 월급 (단위: 달러)
- OverTime: 야근 여부 (Yes, No)
- PercentSalaryHike: 전년 대비 급여 인상율(단위: %)
- TotalWorkingYears: 총 경력 연수

## **2. 모델링 1**

- 다양한 구조의 모델 2개 이상을 설계합니다.
- Hidden Layer, Node 수 조절

### (1) 데이터 전처리

#### 1) 데이터 준비

In [ ]:
# 불필요한 변수 제거
data.drop(columns='EmployeeNumber', inplace=True)

In [ ]:
# x, y 분리
target = 'Attrition'
x = data.drop(columns=target)
y = data.loc[:, target]

#### 2) 가변수화

- 범주형 데이터이면서 값이 0,1 로 되어 있는 것이 아니라면, 가변수화를 수행해야 합니다.
- 대상이 되는 변수에 대해서 가변수화를 수행합니다.

In [ ]:
# 가변수화
dum_cols = ['Gender', 'MaritalStatus', 'OverTime']

x = pd.get_dummies(x, columns=dum_cols, drop_first=True, dtype=int)
x.head()

#### 3) 데이터 분할

In [ ]:
# 학습용, 검증용 분리
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=1)

#### 4) 스케일링

In [ ]:
# 스케일링
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)

### (2) 모델링

#### 1) 모델 선언

In [ ]:
# 메모리 정리
clear_session()

# 입력 Feature 수
nfeatures = x.shape[1]

# Sequential 모델 선언
model = Sequential([
    Input(shape=(nfeatures, )),
    Dense(1, activation='sigmoid')
])

# 모델 요약
model.summary()

#### 2) 모델 학습

In [ ]:
# 학습 설정
model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy')

In [ ]:
# 모델 학습
hist = model.fit(x_train, y_train, epochs=100, validation_split=0.2, verbose=0).history

In [ ]:
# 학습 곡선
dl_history_plot(hist)

#### 3) 예측 및 성능 평가

In [ ]:
# 예측
y_pred = model.predict(x_val)
y_pred = np.where(y_pred >= 0.5, 1, 0)

In [ ]:
# 성능 평가
print(classification_report(y_val, y_pred))

## **3. 모델링 2**

- 다양한 구조의 모델을 설계합니다.
- Hidden Layer, Node 수 조절

#### 1) 모델 선언

In [ ]:
# 메모리 정리
clear_session()

# 입력 Feature 수
nfeatures = x.shape[1]

# Sequential 모델 선언
model = Sequential([
    Input(shape=(nfeatures, )),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(4, activation='relu'),
    Dense(1, activation='sigmoid')
])

# 모델 요약
model.summary()

#### 2) 모델 학습

In [ ]:
# 학습 설정
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy')

In [ ]:
# 모델 학습
hist = model.fit(x_train, y_train, epochs=150, validation_split=0.2, verbose=0).history

In [ ]:
# 학습 곡선
dl_history_plot(hist)

#### 3) 예측 및 모델 평가

In [ ]:
# 예측
y_pred = model.predict(x_val)
y_pred = np.where(y_pred >= 0.5, 1, 0)

In [ ]:
# 성능 평가
print(classification_report(y_val, y_pred))

## **4. 모델링 3: Resampling**

- 불균형 클래스이므로 언더 샘플링 후 모델링합니다.

#### 1) Resampling

In [ ]:
# 함수 불러오기
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
# 언더 샘플링
rus = RandomUnderSampler()
x_train_rus, y_train_rus = rus.fit_resample(x_train, y_train)

#### 2) 모델 선언

In [ ]:
# 메모리 정리
clear_session()

# 입력 Feature 수
nfeatures = x.shape[1]

# Sequential 모델 선언
model = Sequential([
    Input(shape=(nfeatures, )),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(4, activation='relu'),
    Dense(1, activation='sigmoid')
])

# 모델 요약
model.summary()

#### 3) 모델 학습

In [ ]:
# 학습 설정
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy')

In [ ]:
# 모델 학습
hist = model.fit(x_train_rus, y_train_rus, epochs=150, validation_split=0.2, verbose=0).history

In [ ]:
# 학습 곡선
dl_history_plot(hist)

#### 4) 예측 및 성능 평가

In [ ]:
# 예측
y_pred = model.predict(x_val)
y_pred = np.where(y_pred >= 0.5, 1, 0)

In [ ]:
# 평가
print(classification_report(y_val, y_pred))